# P2 — Baseline Training

Notebook này chuẩn bị P1 nếu chưa hoàn tất, rồi chạy P2 trong **cùng một tiến trình Python và cùng experiment ID**.
Chỉ có một cell thực thi pipeline; không chạy P3–P5 và không đánh giá Test.

Giữ `MODE = 'FULL'` và `BANK_EXTENSION = False` cho run hiện tại.
Nếu P1/P2 đã PASS, notebook kiểm tra checksum và giữ artifact đã có.
Nếu runtime đổi phiên bản, một experiment ID mới có thể được tạo; dữ liệu gốc và run cũ được giữ nguyên.
Đây là xử lý cho notebook 2; chưa thay đổi cơ chế ID khi chuyển sang notebook khác.

Sau lần cài dependencies đầu tiên, restart runtime nếu Colab yêu cầu, rồi chạy lại từ trên xuống.


In [1]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/FJCAI2027_PCRR').resolve()
assert (ROOT / 'data/raw/dataset_manifest.json').is_file(), 'Missing existing project/manifest'
assert (ROOT / 'run_all.py').is_file(), 'Wait for project files to sync to Drive'
probe = ROOT / '.colab_write_probe'
probe.write_text('ok'); probe.unlink()
print(ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR


In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements.txt')], check=True)
print('After initial install, restart runtime, then rerun mount and remaining cells.')


After initial install, restart runtime, then rerun mount and remaining cells.


In [3]:
MODE = 'FULL'  # SMOKE -> PILOT -> FULL
BANK_EXTENSION = True  # Set True consistently for all FULL stages to include Bank
import os
os.environ['PCRR_ROOT'] = str(ROOT)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'


In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(ROOT/'tests/test_core.py')], cwd=ROOT, check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', '/content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR/tests/test_core.py'], returncode=0)

In [5]:
import subprocess, sys

# One process, one Experiment instance: P1 -> P2 only.
PIPELINE_CODE = "import sys\nfrom pathlib import Path\nfrom src.experiment import Experiment\nfrom src.io import read_json\n\nexp = Experiment(Path(sys.argv[1]), sys.argv[2], sys.argv[3] == 'true')\nprint(f'Experiment: {exp.id}', flush=True)\nfor stage in ('P1', 'P2'):\n    if exp.state[stage] == 'PASS':\n        exp.verify_hashes(read_json(exp.out / f'{stage.lower()}_hashes.json'))\n        print(f'{stage}: PASS (existing artifacts verified; skipped)', flush=True)\n    else:\n        print(f'{stage}: starting', flush=True)\n        exp.run(stage)\n        print(f'{stage}: {exp.state[stage]}', flush=True)\nprint('PCRR_RUN_DIR=' + str(exp.out), flush=True)\n"
command = [sys.executable, '-u', '-c', PIPELINE_CODE,
           str(ROOT), MODE, str(BANK_EXTENSION).lower()]
result = subprocess.run(command, cwd=ROOT, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()
RUN_DIR = Path(next(line.split('=', 1)[1] for line in result.stdout.splitlines()
                    if line.startswith('PCRR_RUN_DIR=')))


Experiment: full-62500e11e892b899
P1: starting
P1: PASS
P2: starting
P2: PASS
PCRR_RUN_DIR=/content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR/results/full-62500e11e892b899



In [6]:
from IPython.display import Markdown, display
report = ROOT / 'reports' / RUN_DIR.name / 'P2_ACCEPTANCE.md'
assert report.is_file(), f'Missing report: {report}'
display(Markdown(report.read_text(encoding='utf-8')))
print('Completed P1/P2 for:', RUN_DIR.name)
print('Artifacts:', RUN_DIR)
print('This notebook does not run P3, P4 or P5.')


# P2 Acceptance — full-62500e11e892b899

Status: **PASS**

## Work and evidence

45 models loaded and validated on Model-Val. No Test predictions.

## Artifacts

- results/full-62500e11e892b899/
- data/processed/full-62500e11e892b899/
- checkpoints/full-62500e11e892b899/

## Verification

`python -m pytest -q`

Stage command: `python run_all.py --mode FULL --stage P2`

Stage PASS refers to executed runtime assertions; unit-test evidence is recorded separately in reports/IMPLEMENTATION.md.

## Limits and next gate

SMOKE/PILOT are development-only, not publication evidence. P5 requires FULL and verified P4 artifacts. Scientific acceptance requires inspection of actual results.


Completed P1/P2 for: full-62500e11e892b899
Artifacts: /content/drive/.shortcut-targets-by-id/1QhtXC4aI-EzmwXaLnGF94snRLAVx_HA5/FJCAI2027_PCRR/results/full-62500e11e892b899
This notebook does not run P3, P4 or P5.
